In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import spearmanr

from lib import PROBLEM_OVERVIEW_PATH

In [ ]:
problem_df = pd.read_csv(PROBLEM_OVERVIEW_PATH)
problem_df.describe()

In [ ]:
df = problem_df[
    [
        "n_conditions",
        "n_est_parameters",
        "n_measurements",
        "n_observables",
        "amici_nx_solver",
    ]
]
df = df.rename(
    columns={
        "n_conditions": "# Conditions",
        "n_est_parameters": "# Est. parameters",
        "n_measurements": "# Measurements",
        "n_observables": "# Observables",
        "amici_nx_solver": "# State variables",
    }
)

g = sns.pairplot(
    df,
)

g.set(xscale="log", yscale="log")

# sns.pairplot with corner=True removes y-axis decoration for the diagonal
#  remove the upper triangle manually
for i in range(len(g.axes)):
    for j in range(len(g.axes[i])):
        if j > i:
            g.axes[i][j].set_visible(False)

# we want equal log bins for all variables, but sns.pairplot does not support this out of the box
for ax, col in zip(g.diag_axes, df.columns, strict=False):
    x = df[col]

    # variable-specific log bins
    bins = np.logspace(np.log10(x.min()), np.log10(x.max() + 1), 10)

    # remove original histogram, but keep other ax settings
    for patch in list(ax.patches):
        patch.remove()

    ax.hist(x, bins=bins)

    ax.set_ylabel("# Problems")
    ax.yaxis.label.set_visible(True)


def p_stars(p: float) -> str:
    if p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "n.s."


# annotation for spearman correlation and p-value
for i in range(len(g.axes)):
    for j in range(len(g.axes[i])):
        if j < i:
            x = df.iloc[:, j]
            y = df.iloc[:, i]
            corr, p = spearmanr(x, y)
            ax = g.axes[i][j]
            ax.annotate(
                f"$\\rho_S = {corr:.2f}$\n$p=${p:.2e} ({p_stars(p)})",
                xy=(0.02, 0.75),
                xycoords="axes fraction",
            )

plt.savefig("out/FigureS1.pdf")
plt.show()